# Gemini API Basics: Direct Model Access with `google-genai`

## 📚 Learning Objectives

In this notebook, you will learn how to:
- **Call the Gemini API directly** with Google's current first-party SDK, `google-genai` (`from google import genai`)
- **Send multimodal input** — text plus an image — in a single request
- **Expose a Python function to Gemini as a tool** and walk the manual function-calling loop (request → execute → respond)
- **Request schema-constrained structured output** and get back a parsed Python object instead of raw text

## 🎯 Where This Fits

This notebook lives in `06_Agent_SDKs_First_Party/Google_AI_SDK/01_Foundations/` — the first-party, general-purpose **Gemini/GenAI SDK** track. It is deliberately **not** an agent framework: there are no planning loops, no multi-agent orchestration, no `Runner`/`Session` abstractions. It is the direct-model-access layer, comparable in role to instantiating `ChatOpenAI` directly rather than building a LangGraph agent around it. The sibling track, `06_Agent_SDKs_First_Party/Google_ADK/`, covers Google's actual agent-building framework (`google-adk`) — a different package with a different job, built *on top of* the same underlying Gemini models this notebook talks to directly.

This notebook uses `google-genai`'s own native client and config objects, not this repo's `helpers.get_llm()` factory (that factory is reserved for LangGraph-phase notebooks) — the same convention the `Google_ADK` notebook follows for its own native SDK.

## 🔑 Key Concepts

- **`google-genai`**: Google's current, unified Python SDK for the Gemini API (works against both the plain Gemini Developer API and Vertex AI). This supersedes the older, now-deprecated `google-generativeai` package — always prefer `google-genai` (`from google import genai`) in new code.
- **`genai.Client`**: The single entry point for all API calls. Authenticates from a `GOOGLE_API_KEY` env var automatically when no explicit key/project is passed.
- **`client.models.generate_content(...)`**: The core call for one-shot (non-chat) generation — takes a `model` id string and `contents` (text, images, or a mix), plus an optional `config` (`types.GenerateContentConfig`) for temperature, tools, response schema, and more.
- **Multimodal `contents`**: A plain Python list can mix a string prompt with a `PIL.Image.Image` object directly — no manual base64 encoding required.
- **Function calling / tools**: A `GenerateContentConfig(tools=[...])` can take plain Python functions directly (the SDK auto-generates the schema and, by default, even auto-executes the call for you — "automatic function calling"). We'll disable that default here so we can walk the manual request/execute/respond loop explicitly, since seeing that loop once is what makes tool use in *any* SDK or framework legible.
- **Structured output**: `GenerateContentConfig(response_mime_type="application/json", response_schema=<a Pydantic model or type>)` constrains the model's output to that schema; the parsed object comes back on `response.parsed`.

## 🚀 Let's Get Started!

## 1. Setup: Installing `google-genai` and Creating a Client

### What we are going to do

We install `google-genai` (verified installed version: **1.26.0** at the time this notebook was authored) and create a `genai.Client`. The client reads `GOOGLE_API_KEY` from the environment automatically — the same env var this repo's `CLAUDE.md` already documents as a required variable (it's also used elsewhere in this repo for LangExtract Streamlit apps), so no new secret needs to be provisioned.

> ⚠️ **Package name matters.** Google has shipped three different Python packages for Gemini access over time. This notebook uses **`google-genai`** (import path `from google import genai`) — the current, actively-developed, unified SDK. It is *not* the older, deprecated `google-generativeai` package, and it is *not* `google-adk` (that's the agent-*framework* package covered in the sibling `Google_ADK/` track — a different job entirely).

In [ ]:
# ================================================================================
# SETUP: Install google-genai
# ================================================================================
# google-genai is Google's current, unified Python SDK for direct Gemini access
# (Gemini Developer API and Vertex AI, via the same client). It is NOT the older
# deprecated `google-generativeai` package, and it is NOT `google-adk` (the
# agent-framework package covered in the sibling Google_ADK/ track).
# ================================================================================

!pip install -q google-genai pillow matplotlib

In [ ]:
# ================================================================================
# SETUP: Imports, environment, and client
# ================================================================================
# GOOGLE_API_KEY is already a documented env var for this repo (see CLAUDE.md,
# where it is used for LangExtract Streamlit apps). genai.Client() picks it up
# from the environment automatically -- no explicit api_key= argument needed.
# ================================================================================

import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

assert os.getenv("GOOGLE_API_KEY"), (
    "Set GOOGLE_API_KEY in your .env file (a plain Gemini API key from Google AI "
    "Studio works -- no GCP project or Vertex AI setup required for this notebook)."
)

client = genai.Client()
MODEL_ID = "gemini-2.0-flash"

print(f"Client ready. Default model for this notebook: {MODEL_ID}")

### Discussion of the Output

Creating a `Client` performs no network call by itself — it just configures how requests will be authenticated and sent, the same "build vs. run" separation you've seen with ADK's `Agent` or a LangGraph graph. Every actual API call in this notebook goes through `client.models.generate_content(...)`.

## 2. Basic Text Generation

### What we are going to do

The simplest possible call: `client.models.generate_content(model=..., contents=...)` with a plain string prompt. This is the direct analogue of calling `ChatOpenAI(...).invoke(...)` or `client.chat.completions.create(...)` in other SDKs — no agent loop, no tools, just one request and one response.

In [ ]:
# ================================================================================
# STEP 1: A single text-only generation call
# ================================================================================
# `contents` accepts a plain string for the simplest case. The response object
# exposes a convenient `.text` shortcut that concatenates the text parts of the
# first candidate.
# ================================================================================

response = client.models.generate_content(
    model=MODEL_ID,
    contents="In one sentence, what is the Gemini API good for that a smaller model isn't?",
)

print(response.text)

### Discussion of the Output

`response.text` is a convenience property over `response.candidates[0].content.parts[...]`. For anything beyond plain text (function calls, multiple parts), you'll want to inspect `response.candidates[0].content.parts` directly, as the next sections do.

## 3. Multimodal Input: Text + Image

### What we are going to do

`google-genai` accepts a `PIL.Image.Image` object directly inside the `contents` list, mixed with a plain string — no manual base64 encoding or MIME-type wrangling required. To keep this notebook fully self-contained (no external file downloads), we generate a small test image in-memory with `matplotlib`, decode it into a `PIL.Image`, and hand both the image and a text instruction to Gemini in a single call.

In [ ]:
# ================================================================================
# STEP 2: Generate a small test image in-memory (no external file needed)
# ================================================================================
# We draw a simple bar chart with matplotlib, save it to an in-memory buffer as
# PNG bytes, then load those bytes into a PIL.Image -- the object type
# google-genai accepts directly inside `contents`.
# ================================================================================

import io

import matplotlib.pyplot as plt
from PIL import Image

fig, ax = plt.subplots(figsize=(4, 3))
categories = ["Text", "Image", "Function\nCalling", "Structured\nOutput"]
values = [4, 3, 5, 2]
ax.bar(categories, values, color="#4285F4")
ax.set_title("Demo Gemini Capabilities")
ax.set_ylabel("Arbitrary score")
fig.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png")
plt.close(fig)
buf.seek(0)

test_image = Image.open(buf)
print(f"Generated in-memory test image: {test_image.size[0]}x{test_image.size[1]} px, mode={test_image.mode}")

In [ ]:
# ================================================================================
# STEP 3: Send the image + a text instruction in one multimodal request
# ================================================================================
# `contents` is a list mixing a PIL.Image.Image and a plain string. Gemini reads
# both and answers grounded in what it sees in the chart.
# ================================================================================

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        test_image,
        "This is a bar chart. Describe what it shows in 1-2 sentences, and name "
        "which category has the highest bar.",
    ],
)

print(response.text)

### Discussion of the Output

The model should correctly identify "Function Calling" as the tallest bar and describe the chart as comparing scores across the four labeled categories — evidence it actually processed pixel content, not just the surrounding text. This same `contents=[image, text]` pattern extends to multiple images, audio, and video files (passed as `types.Part.from_bytes(...)` or uploaded via `client.files.upload(...)` for larger files) — a single, consistent API surface regardless of modality.

## 4. Function Calling: Manual Request → Execute → Respond Loop

### What we are going to do

We define one plain Python function, expose it to Gemini via `types.GenerateContentConfig(tools=[...])`, and walk the **manual** function-calling loop step by step:

1. Send a prompt that should require the tool.
2. Inspect `response.function_calls` — Gemini doesn't call our function itself; it returns a *request* to call it, with arguments it inferred from the prompt.
3. Execute the function ourselves in Python.
4. Send the function's result back to Gemini as a `types.Part.from_function_response(...)`, alongside the conversation history so far.
5. Get the final, natural-language answer.

> By default, `google-genai` supports *automatic* function calling (pass a plain Python function directly as a tool and the SDK executes it and loops for you, similar to ADK's tool auto-wrapping). We turn that off here (`automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)`) purely so the request/execute/respond mechanics are visible — the same mechanics automatic mode just hides behind one call.

In [ ]:
# ================================================================================
# STEP 4: Define a plain Python function as a tool
# ================================================================================
# google-genai can auto-generate a function's schema from its signature, type
# hints, and docstring -- no manual JSON-schema authoring required, the same
# convenience you saw with ADK's tool wrapping.
# ================================================================================


def get_current_weather(city: str) -> dict:
    """Looks up the current weather for a city.

    Args:
        city: The name of the city, e.g. "Paris".

    Returns:
        A dict with "city", "condition", and "temperature_celsius" keys.
    """
    # Stubbed lookup table standing in for a real weather API call.
    fake_weather_db = {
        "paris": {"condition": "cloudy", "temperature_celsius": 18},
        "tokyo": {"condition": "sunny", "temperature_celsius": 27},
    }
    key = city.strip().lower()
    data = fake_weather_db.get(key, {"condition": "unknown", "temperature_celsius": None})
    return {"city": city, **data}

In [ ]:
# ================================================================================
# STEP 5: Ask a question that should trigger a tool call
# ================================================================================
# We disable automatic function calling so Gemini's tool-call *request* comes
# back to us instead of being auto-executed by the SDK.
# ================================================================================

tool_config = types.GenerateContentConfig(
    tools=[get_current_weather],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

user_prompt = "What's the weather like in Tokyo right now?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_prompt,
    config=tool_config,
)

requested_calls = response.function_calls
print("Requested function calls:", requested_calls)

### Discussion of the Output

`response.function_calls` returns a list of `types.FunctionCall` objects (name + args) rather than any text — Gemini decided the prompt needs `get_current_weather(city="Tokyo")` and is asking us to run it and report back, instead of guessing an answer.

In [ ]:
# ================================================================================
# STEP 6: Execute the function ourselves, then send the result back
# ================================================================================
# We build the full conversation history: the original user turn, the model's
# function-call turn, and a new "tool result" turn -- then call generate_content
# again so Gemini can produce its final natural-language answer.
# ================================================================================

call = requested_calls[0]
tool_result = get_current_weather(**call.args)
print("Tool executed locally, result:", tool_result)

conversation = [
    types.Content(role="user", parts=[types.Part(text=user_prompt)]),
    response.candidates[0].content,  # the model's turn requesting the call
    types.Content(
        role="user",
        parts=[types.Part.from_function_response(name=call.name, response=tool_result)],
    ),
]

final_response = client.models.generate_content(
    model=MODEL_ID,
    contents=conversation,
    config=tool_config,
)

print("\n--- Final answer ---")
print(final_response.text)

### Discussion of the Output

The final answer should mention Tokyo's sunny condition and 27°C — information that only exists in our stubbed `get_current_weather` function, never in the model's training data or the prompt itself. This request → execute → respond cycle is the same mechanism every agent framework's "tool use" is built on top of (LangGraph's `ToolNode`, ADK's automatic tool wrapping, CrewAI's tool calling) — here we're seeing it with nothing in between.

## 5. Structured Output: Schema-Constrained JSON

### What we are going to do

Instead of parsing free-form text, we ask Gemini to return output matching a specific schema by setting `response_mime_type="application/json"` and `response_schema=<a Pydantic model>` on `GenerateContentConfig`. The SDK constrains generation to that schema and — as a convenience — hands back an already-parsed Python object on `response.parsed`, so no manual `json.loads` + validation step is needed.

In [ ]:
# ================================================================================
# STEP 7: Define a Pydantic schema and request schema-constrained output
# ================================================================================
# response_schema accepts a Pydantic model directly. Combined with
# response_mime_type="application/json", Gemini's output is constrained to
# valid JSON matching this schema.
# ================================================================================

from pydantic import BaseModel


class RecipeIdea(BaseModel):
    name: str
    cuisine: str
    main_ingredients: list[str]
    estimated_minutes: int


structured_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=RecipeIdea,
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Suggest one quick weeknight dinner idea.",
    config=structured_config,
)

recipe: RecipeIdea = response.parsed
print(type(recipe))
print(recipe)
print("\nCuisine only:", recipe.cuisine)

### Discussion of the Output

`response.parsed` is already a validated `RecipeIdea` instance, not a raw string — attribute access like `recipe.cuisine` works immediately. `response.text` still holds the underlying raw JSON string if you need it. This is the same schema-first discipline you'd get from LangChain's `.with_structured_output(...)` or OpenAI's `response_format`, expressed through this SDK's own `response_schema` config field.

## 📖 Key Takeaways

- **Use `google-genai`, not `google-generativeai`.** `from google import genai` is Google's current, actively-maintained SDK for direct Gemini access (verified installed version 1.26.0 for this notebook); the older `google-generativeai` package is deprecated, and `google-adk` is a separate agent-*framework* package (covered in the sibling `Google_ADK/` track).
- **`genai.Client()` + `client.models.generate_content(model=..., contents=...)`** is the whole core API surface for one-shot generation — it reads `GOOGLE_API_KEY` from the environment automatically.
- **Multimodal input is just a list.** `contents=[a_pil_image, "some text"]` mixes an image and a prompt in one call, no manual encoding required.
- **Function calling has a manual loop underneath automatic mode**: Gemini returns a `FunctionCall` request via `response.function_calls`, you execute it in your own Python code, and you feed the result back as a `types.Part.from_function_response(...)` turn to get the final answer. Automatic mode (the SDK's default when you pass a plain function as a tool) just hides this loop behind one call.
- **Structured output** comes from `GenerateContentConfig(response_mime_type="application/json", response_schema=<a Pydantic model>)`, with the validated object available directly on `response.parsed`.

### How this notebook differs from `Google_ADK/`

This notebook is **direct model access** — the same layer as instantiating `ChatOpenAI` or `ChatAnthropic` directly: one client, one call, one response, with no persistent agent state. There is no planning loop, no multi-turn `Runner`/`Session` harness, no multi-agent orchestration, and no automatic tool-execution *agent* wrapped around the model (only the SDK-level automatic function-calling convenience, which we deliberately turned off above to show the mechanics).

`06_Agent_SDKs_First_Party/Google_ADK/` sits one layer up: it builds an `Agent` (`LlmAgent`) around a Gemini model, drives it through a `Runner` + `SessionService` event loop, and is designed for multi-step, tool-using, potentially multi-agent behavior — the same relationship as "calling an LLM API directly" vs. "building a LangGraph agent around that same LLM."

### 🎓 Next Steps

- `06_Agent_SDKs_First_Party/Google_AI_SDK/02_Core_Capabilities/` will cover function calling and structured output in more depth, plus streaming responses.
- Compare the manual function-calling loop here to ADK's automatic tool-wrapping in `Google_ADK/01_Foundations/01_Google_ADK_and_AgentSpace_Basics.ipynb` — same underlying mechanism, different levels of abstraction.
- Try `client.aio.models.generate_content(...)` for the async equivalent, or `client.models.generate_content_stream(...)` for token-by-token streaming.

### 📚 Additional Resources

- [google-genai on PyPI](https://pypi.org/project/google-genai/)
- [google-genai GitHub Repository](https://github.com/googleapis/python-genai)
- [Gemini API Documentation](https://ai.google.dev/gemini-api/docs)